# VehiAlpes — Capa Gold: dimensiones

MINE-4214 · Taller 1 · Punto 2

## Objetivo de la capa

Materializar el modelo dimensional del punto 1: seis dimensiones conformadas con
llaves surrogadas, listas para ser consultadas por el tablero sin transformaciones
adicionales.

## Decisión transversal: por qué llaves surrogadas

Las llaves naturales no sirven como llave foránea de los hechos en este caso, y no
por purismo:

- **La placa no identifica una fila de `dim_vehiculo`.** Cada placa tiene entre dos
  y cuatro versiones, y el hecho debe apuntar a la versión vigente en la fecha del
  evento. Con la placa como FK, el join devolvería varias filas y la tarifa
  histórica sería irrecuperable.
- **El `id_cliente` no identifica una persona.** 60 documentos tienen dos
  identificadores.

Este es exactamente el error del ejercicio de `Dim_Cliente` visto en clase: dos
filas con la misma llave primaria, ambas vigentes, y un hecho que no puede
discriminar entre ellas.

## Decisión: miembros centinela

Toda dimensión lleva una fila con llave `-1` para "no aplica" y `-2` para
"desconocido". Esto permite que **ninguna llave foránea de los hechos sea nula**,
que es la convención Kimball. Es indispensable en el ciclo de vida: 350 de 500
vehículos no tienen fecha de venta, y sin centinela ese join se rompería en
silencio.

In [0]:
from pyspark.sql import functions as F, Window as W

CATALOGO = "vehialpes"
SK_NO_APLICA = -1
SK_DESCONOCIDO = -2

## dim_fecha

Se genera por calendario, no desde los datos. Generarla desde las fechas
observadas dejaría huecos en los días sin transacciones, y el snapshot de ocupación
necesita **todos** los días para producir una fila por vehículo-día.

La llave surrogada es la fecha en formato `AAAAMMDD` en lugar de un secuencial. Es
legible al depurar y permite filtrar rangos sin hacer join contra la dimensión.

In [0]:
FECHA_MIN, FECHA_MAX = "2022-12-01", "2026-12-31"

dim_fecha = (spark.sql(f"""
        SELECT explode(sequence(
            to_date('{FECHA_MIN}'), to_date('{FECHA_MAX}'), interval 1 day)) AS fecha""")
    .withColumn("sk_fecha", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("nombre_mes", F.date_format("fecha", "MMMM"))
    .withColumn("anio_mes", F.date_format("fecha", "yyyy-MM"))
    .withColumn("dia", F.dayofmonth("fecha"))
    .withColumn("num_semana_anio", F.weekofyear("fecha"))
    .withColumn("dia_semana", F.date_format("fecha", "EEEE"))
    .withColumn("es_fin_de_semana", F.dayofweek("fecha").isin(1, 7))
    .withColumn("es_dia_habil", ~F.dayofweek("fecha").isin(1, 7)))

centinelas_fecha = spark.createDataFrame(
    [(SK_NO_APLICA, "(no aplica)"), (SK_DESCONOCIDO, "(desconocido)")],
    "sk_fecha int, nombre_mes string")

(dim_fecha.unionByName(centinelas_fecha, allowMissingColumns=True)
    .write.mode("overwrite").saveAsTable(f"{CATALOGO}.gold.dim_fecha"))

### Vistas de role-playing

El alquiler usa la fecha en dos papeles (entrega y devolución) y el ciclo de vida en
cuatro hitos. En lugar de duplicar la tabla, se crean vistas con alias.

Duplicar la dimensión física sería el error: dos copias de la misma tabla se
desincronizan al mantenerla, y el criterio de calidad pide nombres precisos y claros
para la empresa. Las vistas renombran las columnas de forma que el tablero muestre
"año de entrega" y "año de devolución" sin ambigüedad.

In [0]:
PAPELES = {
    "v_fecha_entrega": "entrega",
    "v_fecha_devolucion": "devolucion",
    "v_fecha_compra": "compra",
    "v_fecha_venta": "venta",
}

COLUMNAS_FECHA = ["fecha", "anio", "trimestre", "mes", "nombre_mes", "anio_mes",
                  "dia", "num_semana_anio", "dia_semana", "es_fin_de_semana"]

for vista, sufijo in PAPELES.items():
    proyeccion = [f"sk_fecha AS sk_fecha_{sufijo}"] + \
                 [f"{c} AS {c}_{sufijo}" for c in COLUMNAS_FECHA]
    spark.sql(f"""
        CREATE OR REPLACE VIEW {CATALOGO}.gold.{vista} AS
        SELECT {', '.join(proyeccion)} FROM {CATALOGO}.gold.dim_fecha""")

## dim_vehiculo (SCD tipo 2)

Las vigencias ya se construyeron y validaron en silver. Aquí solo se asigna la
llave surrogada.

**Por qué `row_number` ordenado por placa y vigencia.** La llave debe ser estable
entre ejecuciones para que los hechos ya cargados sigan apuntando a la versión
correcta. Ordenar por `(placa, fe_vig_ini)` es determinista; usar
`monotonically_increasing_id` no lo es y rompería los hechos en cada recarga.

En un pipeline productivo esta asignación se haría con `MERGE` incremental para no
reasignar llaves existentes. Se deja el `row_number` por claridad didáctica, y se
señala explícitamente como la decisión a cambiar al productivizar.

In [0]:
ventana_sk = W.orderBy(F.col("placa").asc(), F.col("fe_vig_ini").asc())

dim_vehiculo = (spark.table(f"{CATALOGO}.silver.dim_vehiculo_vigencias")
    .withColumn("sk_vehiculo", F.row_number().over(ventana_sk))
    .select(
        "sk_vehiculo", "placa", "marca", "modelo", "anio_modelo", "tipo_combustible",
        "color", "costo_alquiler_dia", "valor_seguro", "costo_mantenimiento_km",
        "fecha_ingreso_concesionario", "fe_vig_ini", "fe_vig_fin",
        "es_version_actual", "num_version",
        "ind_anio_imputado", "ind_combustible_imputado", "ind_color_faltante",
        "ind_tarifa_fuera_de_rango"))

centinela_vehiculo = spark.createDataFrame(
    [(SK_NO_APLICA, "(no aplica)"), (SK_DESCONOCIDO, "(desconocido)")],
    "sk_vehiculo int, placa string")

(dim_vehiculo.unionByName(centinela_vehiculo, allowMissingColumns=True)
    .write.mode("overwrite").saveAsTable(f"{CATALOGO}.gold.dim_vehiculo"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## dim_cliente (SCD tipo 7)

La autorreferencia `sk_cliente_unificado` es lo que resuelve los 60 duplicados.
Se asigna en dos pasos porque la columna apunta a la misma tabla: primero se
generan las llaves, luego se resuelve la referencia contra las llaves ya asignadas.

El beneficio concreto: un reporte de ingresos por cliente puede agrupar por
`sk_cliente_unificado` y obtener la cifra por persona real, o por `sk_cliente` y
obtener la cifra por registro del CRM. Las dos lecturas son legítimas y el modelo
no fuerza a escoger una.

In [0]:
base_cliente = (spark.table(f"{CATALOGO}.silver.clientes")
    .withColumn("sk_cliente", F.row_number().over(W.orderBy(F.col("id_cliente").asc()))))

mapa_sk = base_cliente.select(
    F.col("id_cliente").alias("_id"), F.col("sk_cliente").alias("_sk"))

dim_cliente = (base_cliente
    .join(mapa_sk, F.col("id_cliente_unificado") == F.col("_id"), "left")
    .withColumn("sk_cliente_unificado", F.coalesce(F.col("_sk"), F.col("sk_cliente")))
    .select(
        "sk_cliente", "id_cliente", "sk_cliente_unificado", "numero_documento",
        "nombres", "apellidos", "nombre_completo", "nombre_canonico",
        "ind_registro_duplicado", "es_miembro_inferido"))

centinela_cliente = spark.createDataFrame(
    [(SK_NO_APLICA, "(no aplica)"), (SK_DESCONOCIDO, "(desconocido)")],
    "sk_cliente int, nombre_completo string")

(dim_cliente.unionByName(centinela_cliente, allowMissingColumns=True)
    .write.mode("overwrite").saveAsTable(f"{CATALOGO}.gold.dim_cliente"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## dim_sucursal y dim_proveedor

**Sucursal: la ciudad va denormalizada dentro de la dimensión.** La jerarquía
sucursal–ciudad es 1:1 (siete sucursales, siete ciudades, sin cruces). Crear una
`dim_ciudad` aparte produciría un copo de nieve, que el criterio de calidad
descarta explícitamente para jerarquías de atributos.

**Proveedor: dimensión de un solo atributo, y se justifica.** Seis filas extraídas
del texto de la transacción porque no hay maestro. Existe porque "qué carros
comprar" es una pregunta central del caso y el análisis por proveedor es parte de la
respuesta. Si VehiAlpes entregara un maestro con ciudad, tipo y condiciones
comerciales, esta dimensión ganaría mucho valor sin cambiar el modelo.

In [0]:
sv = spark.table(f"{CATALOGO}.silver.transacciones")

dim_sucursal = (sv.select("nombre_sucursal", "ciudad").distinct()
    .filter(F.col("nombre_sucursal").isNotNull())
    .withColumn("sk_sucursal", F.row_number().over(W.orderBy("nombre_sucursal"))))

(dim_sucursal.unionByName(
    spark.createDataFrame([(SK_NO_APLICA, "(no aplica)")],
                          "sk_sucursal int, nombre_sucursal string"),
    allowMissingColumns=True)
 .write.mode("overwrite").saveAsTable(f"{CATALOGO}.gold.dim_sucursal"))

dim_proveedor = (sv.select("nombre_proveedor").distinct()
    .filter(F.col("nombre_proveedor").isNotNull())
    .withColumn("sk_proveedor", F.row_number().over(W.orderBy("nombre_proveedor"))))

(dim_proveedor.unionByName(
    spark.createDataFrame([(SK_NO_APLICA, "(no aplica)")],
                          "sk_proveedor int, nombre_proveedor string"),
    allowMissingColumns=True)
 .write.mode("overwrite").saveAsTable(f"{CATALOGO}.gold.dim_proveedor"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## dim_pago (dimensión basura)

Combina el método de pago con las banderas de calidad del registro. La alternativa
—una dimensión para el método y tres columnas booleanas en cada hecho— violaría el
criterio de que los atributos del hecho sean llaves de dimensión o medidas.

Se enumeran todas las combinaciones posibles (4 métodos × 2³ banderas = 32 filas)
en lugar de solo las observadas. Así una combinación nueva no obliga a insertar
filas en la dimensión durante la carga del hecho.

In [0]:
metodos = sv.select("metodo_pago").distinct().filter(F.col("metodo_pago").isNotNull())
booleanos = spark.createDataFrame([(True,), (False,)], "v boolean")

dim_pago = (metodos
    .crossJoin(booleanos.withColumnRenamed("v", "ind_valor_inconsistente"))
    .crossJoin(booleanos.withColumnRenamed("v", "ind_fecha_reformateada"))
    .crossJoin(booleanos.withColumnRenamed("v", "ind_registro_deduplicado"))
    .withColumn("sk_pago", F.row_number().over(W.orderBy(
        "metodo_pago", "ind_valor_inconsistente",
        "ind_fecha_reformateada", "ind_registro_deduplicado"))))

dim_pago.write.mode("overwrite").saveAsTable(f"{CATALOGO}.gold.dim_pago")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## Validación de las dimensiones

Dos reglas: la llave surrogada es única y toda dimensión tiene al menos un
centinela. Sin lo segundo, los hechos con hitos no ocurridos quedarían con FK nula.

In [0]:
DIMENSIONES = {
    "dim_fecha": "sk_fecha", "dim_vehiculo": "sk_vehiculo",
    "dim_cliente": "sk_cliente", "dim_sucursal": "sk_sucursal",
    "dim_proveedor": "sk_proveedor", "dim_pago": "sk_pago",
}

for tabla, llave in DIMENSIONES.items():
    df = spark.table(f"{CATALOGO}.gold.{tabla}")
    total, distintas = df.count(), df.select(llave).distinct().count()
    assert total == distintas, f"{tabla}: la llave surrogada no es única"
    if tabla != "dim_pago":
        centinelas = df.filter(F.col(llave) < 0).count()
        assert centinelas > 0, f"{tabla}: falta el miembro centinela"
    print(f"{tabla}: {total} filas, llave única")
    spark.sql(f"OPTIMIZE {CATALOGO}.gold.{tabla}")

dim_fecha: 1494 filas, llave única
dim_vehiculo: 1381 filas, llave única
dim_cliente: 1592 filas, llave única
dim_sucursal: 8 filas, llave única
dim_proveedor: 7 filas, llave única
dim_pago: 32 filas, llave única
